# Appendix: Effect of Prompt

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

data = [
    # Prompt Idx=1
    {"Prompt": "Idx=1", "Method": "T2I", "PSNR": 5.6924, "SSIM": 0.0440, "HFEN": 1.1661, "NMSE": 7.8408},
    {"Prompt": "Idx=1", "Method": "ControlNet", "PSNR": 6.9727, "SSIM": 0.0200, "HFEN": 0.7537, "NMSE": 5.5644},
    {"Prompt": "Idx=1", "Method": "LoRA", "PSNR": 11.3711, "SSIM": 0.0041, "HFEN": 1.0951, "NMSE": 2.0025},
    # Prompt Idx=2
    {"Prompt": "Idx=2", "Method": "T2I", "PSNR": 5.8048, "SSIM": 0.0527, "HFEN": 1.1538, "NMSE": 7.2312},
    {"Prompt": "Idx=2", "Method": "ControlNet", "PSNR": 6.7435, "SSIM": 0.0203, "HFEN": 0.7312, "NMSE": 5.6738},
    {"Prompt": "Idx=2", "Method": "LoRA", "PSNR": 11.5731, "SSIM": 0.0041, "HFEN": 1.0971, "NMSE": 1.9774},
    # Prompt Idx=3
    {"Prompt": "Idx=3", "Method": "T2I", "PSNR": 0, "SSIM": 0, "HFEN": 0, "NMSE": 0},
    {"Prompt": "Idx=3", "Method": "ControlNet", "PSNR": 6.6737, "SSIM": 0.0199, "HFEN": 0.7492, "NMSE": 5.8728},
    {"Prompt": "Idx=3", "Method": "LoRA", "PSNR": 11.5357, "SSIM": 0.0028, "HFEN": 1.0869, "NMSE": 1.9412},
]

df = pd.DataFrame(data)

methods = df['Method'].unique()
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#9467bd']

# Metrics to plot in 2x2 grid
metrics = [
    ("PSNR", "PSNR"),
    ("SSIM", "SSIM"),
    ("HFEN", "HFEN"),
    ("NMSE", "NMSE"),
]

# Create 2x2 subplot grid
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[m[0] for m in metrics],
    horizontal_spacing=0.12,
    vertical_spacing=0.15,
)

for idx, (metric_col, ylabel) in enumerate(metrics):
    row = idx // 2 + 1
    col = idx % 2 + 1

    for i, method in enumerate(methods):
        subset = df[df['Method'] == method]
        # Only show legend for the first subplot to avoid duplicates
        show_legend = (idx == 0)
        pos = "bottom center" if (idx == 0 and method == "T2I") or (idx == 2 and method == "LoRA") else "top center"
        fig.add_trace(
            go.Scatter(
                x=subset['Prompt'],
                y=subset[metric_col],
                text=[str(x) for x in subset[metric_col]],
                textposition=pos,#"top center",
                mode='lines+markers+text',
                name=method,
                legendgroup=method,
                showlegend=show_legend,
                line=dict(width=2, color=colors[i % len(colors)]),
                marker=dict(size=8),
                textfont=dict(size=14),
            ),
            row=row, col=col,
        )
    print(min(subset[metric_col]), max(subset[metric_col]), max(subset[metric_col]) * 1.1)
    fig.update_yaxes(range=[df[metric_col].min(), df[metric_col].max() * 1.15], row=row, col=col)
    # fig.update_xaxes(row=row, col=col)

# Apply scientific styling
fig.update_layout(
    template="simple_white",
    font=dict(family="Arial, sans-serif", size=18, color="black"),
    width=950,
    height=750,
    margin=dict(l=40, r=30, t=50, b=80),
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.12,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(255, 255, 255, 0.9)",
        bordercolor="Black",
        borderwidth=1,
        title=dict(text="Method: "),
    ),
)

# Style all axes
fig.update_xaxes(showline=True, linewidth=1, linecolor='black', mirror=True, range=[-0.3, 2.3])
fig.update_yaxes(showline=True, linewidth=1, linecolor='black', mirror=True)

fig.show()

# Appendix: Taper and Reduction Factor

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans'],
    'font.size': 12,
})

# [HFEN, NMSE, PSNR, SSIM])
results = {
    (1.5, 0.15): [1.12327604, 4.31470232, 8.17333407, 0.01159396],
    (1.5, 0.30): [1.17635135, 4.26217291, 8.38361554, 0.01279164],
    (1.5, 0.45): [1.18971173, 2.87144367, 9.61474962, 0.02437975],
    (3.0, 0.15): [1.11916294, 4.06148272, 8.45214701, 0.01212364],
    (3.0, 0.30): [1.13315548, 4.22740788, 8.26584320, 0.01154302],
    (3.0, 0.45): [1.19535980, 4.27024739, 8.27454085, 0.01230299],
    (4.5, 0.15): [1.19350517, 3.71299548, 8.69202471, 0.01342639],
    (4.5, 0.30): [1.18108851, 3.33055344, 8.52624216, 0.01403216],
    (4.5, 0.45): [1.13480293, 4.35439372, 8.12178774, 0.01126660],
}

baseline = [1.45332586, 3.53461895, 8.61258450, 0.01597837]

taus = [1.5, 3.0, 4.5]
rs   = [0.15, 0.30, 0.45]

metric_names = ["HFEN", "NMSE", "PSNR", "SSIM"]
higher_is_better = [False, False, True, True]  # lower = better for HFEN, NMSE, SSIM
grids  = {m: np.zeros((3, 3)) for m in metric_names}
deltas = {m: np.zeros((3, 3)) for m in metric_names}

for i, tau in enumerate(taus):
    for j, r in enumerate(rs):
        vals = results[(tau, r)]
        for k, m in enumerate(metric_names):
            grids[m][i, j]  = vals[k]
            deltas[m][i, j] = vals[k] - baseline[k]


fig, axes = plt.subplots(2, 2, figsize=(11, 9))
fig.subplots_adjust(hspace=0.38, wspace=0.32, left=0.08, right=0.95, top=0.92, bottom=0.10)

for idx, metric in enumerate(metric_names):
    ax = axes[idx // 2][idx % 2]
    data = grids[metric]
    delta = deltas[metric]
    hib = higher_is_better[idx]

    # Color data: positive = improvement (green)
    if hib:
        color_data = delta
    else:
        color_data = -delta  # flip so positive = good

    vmax = max(abs(color_data.min()), abs(color_data.max()))
    if vmax == 0:
        vmax = 1
    norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    # Use a softer diverging colormap
    cmap = plt.cm.RdYlGn
    im = ax.imshow(color_data, cmap=cmap, norm=norm, aspect='auto',
                   interpolation='nearest')

    for i in range(3):
        for j in range(3):
            val = data[i, j]
            d   = delta[i, j]
            sign = "+" if d >= 0 else ""
            is_improvement = (d > 0) if hib else (d < 0)
            arrow = "▲" if is_improvement else "▼"
            # Format strings
            if metric == "SSIM":
                val_str   = f"{val:.4f}"
                delta_str = f"{arrow} {sign}{d:.4f}"
            elif metric == "PSNR":
                val_str   = f"{val:.2f}"
                delta_str = f"{arrow} {sign}{d:.2f}"
            else:
                val_str   = f"{val:.3f}"
                delta_str = f"{arrow} {sign}{d:.3f}"
            # Determine text color based on background luminance
            # Get the cell's background color
            bg_rgba = cmap(norm(color_data[i, j]))
            luminance = 0.299 * bg_rgba[0] + 0.587 * bg_rgba[1] + 0.114 * bg_rgba[2]
            val_color = 'white' if luminance < 0.45 else 'black'
            # Delta colors: green/red with white outline effect via bold on dark bg
            if luminance < 0.45:
                delta_color = '#90ee90' if is_improvement else '#ffaaaa'
            else:
                delta_color = '#1a6b2a' if is_improvement else '#b82020'

            ax.text(j, i - 0.13, val_str, ha='center', va='center',
                    fontsize=13, fontweight='bold', color=val_color)
            ax.text(j, i + 0.22, delta_str, ha='center', va='center',
                    fontsize=10, fontweight='medium', color=val_color)
    # Axis labels & ticks
    ax.set_xticks(range(3))
    ax.set_xticklabels([f"{r}" for r in rs], fontsize=12)
    ax.set_yticks(range(3))
    ax.set_yticklabels([f"{t}" for t in taus], fontsize=12)
    ax.set_xlabel("Taper width (τ)", fontsize=13)
    ax.set_ylabel("Reduction factor (r)", fontsize=13)
    ax.set_title(metric, fontsize=15, fontweight='bold', pad=10)
    # Clean border styling
    for spine in ax.spines.values():
        spine.set_linewidth(1.0)
        spine.set_color('black')
    ax.tick_params(length=0)
# Baseline annotation
fig.text(0.5, 0.015,
         f"Baseline (no DC):  HFEN = {baseline[0]:.3f}   |   "
         f"NMSE = {baseline[1]:.3f}   |   "
         f"PSNR = {baseline[2]:.2f}   |   "
         f"SSIM = {baseline[3]:.4f}",
         ha='center', va='bottom', fontsize=11,
         style='italic', color='#444444')
plt.savefig("./app_lora_dc_eval.png", dpi=200, bbox_inches='tight',
            facecolor='white', edgecolor='none')

In [ ]:
# [HFEN, NMSE, PSNR, SSIM])
results = {
    (1.5, 0.15): [1.05831963, 7.37258596, 5.29564409, 0.00788712],
    (1.5, 0.30): [1.04957132e+00, 9.08313942e+00, 5.11036897e+00, 7.23626493e-03],
    (1.5, 0.45): [1.26679218e+00, 9.53892870e+00, 4.92215714e+00, 7.34424628e-03],
    (3.0, 0.15): [1.08449517e+00, 7.91801786e+00, 5.16935616e+00, 7.02146934e-03],
    (3.0, 0.30): [1.09562365e+00, 8.45939193e+00, 5.22367468e+00, 7.12326257e-03],
    (3.0, 0.45): [1.07523127e+00, 8.87062750e+00, 5.19501028e+00, 7.25591048e-03],
    (4.5, 0.15): [1.08971503e+00, 1.00266214e+01, 4.98309016e+00, 5.97153734e-03],
    (4.5, 0.30): [1.06913316, 6.99581809, 5.29149337, 0.00818841],
    (4.5, 0.45): [1.06311933e+00, 8.72669401e+00, 5.21620550e+00, 7.24419178e-03],
}

baseline = [1.25550029, 7.4492732, 5.33395977, 0.00884299]

taus = [1.5, 3.0, 4.5]
rs   = [0.15, 0.30, 0.45]

metric_names = ["HFEN", "NMSE", "PSNR", "SSIM"]
higher_is_better = [False, False, True, True]  # lower = better for HFEN, NMSE, SSIM
grids  = {m: np.zeros((3, 3)) for m in metric_names}
deltas = {m: np.zeros((3, 3)) for m in metric_names}
for i, tau in enumerate(taus):
    for j, r in enumerate(rs):
        vals = results[(tau, r)]
        for k, m in enumerate(metric_names):
            grids[m][i, j]  = vals[k]
            deltas[m][i, j] = vals[k] - baseline[k]
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
fig.subplots_adjust(hspace=0.38, wspace=0.32, left=0.08, right=0.95, top=0.92, bottom=0.10)

for idx, metric in enumerate(metric_names):
    ax = axes[idx // 2][idx % 2]
    data = grids[metric]
    delta = deltas[metric]
    hib = higher_is_better[idx]
    # Color data: positive = improvement (green)
    if hib:
        color_data = delta
    else:
        color_data = -delta  # flip so positive = good

    vmax = max(abs(color_data.min()), abs(color_data.max()))
    if vmax == 0:
        vmax = 1
    norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    # Use a softer diverging colormap
    cmap = plt.cm.RdYlGn

    im = ax.imshow(color_data, cmap=cmap, norm=norm, aspect='auto',
                   interpolation='nearest')

    # Annotate each cell
    for i in range(3):
        for j in range(3):
            val = data[i, j]
            d   = delta[i, j]
            sign = "+" if d >= 0 else ""

            is_improvement = (d > 0) if hib else (d < 0)
            arrow = "▲" if is_improvement else "▼"
            # Format strings
            if metric == "SSIM":
                val_str   = f"{val:.4f}"
                delta_str = f"{arrow} {sign}{d:.4f}"
            elif metric == "PSNR":
                val_str   = f"{val:.2f}"
                delta_str = f"{arrow} {sign}{d:.2f}"
            else:
                val_str   = f"{val:.3f}"
                delta_str = f"{arrow} {sign}{d:.3f}"
            # Determine text color based on background luminance
            # Get the cell's background color
            bg_rgba = cmap(norm(color_data[i, j]))
            luminance = 0.299 * bg_rgba[0] + 0.587 * bg_rgba[1] + 0.114 * bg_rgba[2]
            val_color = 'white' if luminance < 0.45 else 'black'
            # Delta colors: green/red with white outline effect via bold on dark bg
            if luminance < 0.45:
                delta_color = '#90ee90' if is_improvement else '#ffaaaa'
            else:
                delta_color = '#1a6b2a' if is_improvement else '#b82020'

            ax.text(j, i - 0.13, val_str, ha='center', va='center',
                    fontsize=13, fontweight='bold', color=val_color)
            ax.text(j, i + 0.22, delta_str, ha='center', va='center',
                    fontsize=10, fontweight='medium', color=val_color)
    # Axis labels & ticks
    ax.set_xticks(range(3))
    ax.set_xticklabels([f"{r}" for r in rs], fontsize=12)
    ax.set_yticks(range(3))
    ax.set_yticklabels([f"{t}" for t in taus], fontsize=12)
    ax.set_xlabel("Taper width (τ)", fontsize=13)
    ax.set_ylabel("Reduction factor (r)", fontsize=13)
    ax.set_title(metric, fontsize=15, fontweight='bold', pad=10)
    # Clean border styling
    for spine in ax.spines.values():
        spine.set_linewidth(1.0)
        spine.set_color('black')
    ax.tick_params(length=0)
# Baseline annotation
fig.text(0.5, 0.015,
         f"Baseline (no DC):  HFEN = {baseline[0]:.3f}   |   "
         f"NMSE = {baseline[1]:.3f}   |   "
         f"PSNR = {baseline[2]:.2f}   |   "
         f"SSIM = {baseline[3]:.4f}",
         ha='center', va='bottom', fontsize=11,
         style='italic', color='#444444')
plt.savefig("./app_t2i_dc_eval.png", dpi=200, bbox_inches='tight',
            facecolor='white', edgecolor='none')